# ASTRA Temporal GRU Training (CausalGRUFallback)

## 1. Environment Setup

In [ ]:
import torch
import sys
import os
import json
import numpy as np
import shutil
import zipfile

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU not available. CPU fallback will be slow.")


## 2. Codebase Import

In [ ]:
# Provide option to upload project zip or clone
import ipywidgets as widgets
from IPython.display import display

print("Upload the ASTRA repository zip (e.g., ASTRA.zip) if needed, or clone:")
# !git clone https://github.com/YOUR_ORG/ASTRA.git
# %cd ASTRA
# For local zip:
if not os.path.exists('ml'):
    print("Please ensure the ASTRA codebase is loaded so ml.models.temporal is accessible.")
    print("If uploading, extract it here.")


## 3. Dataset Input

In [ ]:
from google.colab import files

print("Upload temporal_dataset_v1.zip:")
uploaded = files.upload()

if 'temporal_dataset_v1.zip' in uploaded:
    print("Extracting dataset...")
    os.makedirs('/content/data/temporal_v1', exist_ok=True)
    with zipfile.ZipFile('temporal_dataset_v1.zip', 'r') as zip_ref:
        zip_ref.extractall('/content/data/temporal_v1/')
    
    # Check for nested extraction
    if os.path.exists('/content/data/temporal_v1/data/temporal_v1'):
        print("Fixing nested directory structure...")
        import glob
        for f in glob.glob('/content/data/temporal_v1/data/temporal_v1/*'):
            shutil.move(f, '/content/data/temporal_v1/')
        shutil.rmtree('/content/data/temporal_v1/data')
else:
    print("Dataset not uploaded.")


## 4. Dataset Verification

In [ ]:
DATA_DIR = '/content/data/temporal_v1'
META_PATH = os.path.join(DATA_DIR, 'metadata.json')

with open(META_PATH, 'r') as f:
    meta = json.load(f)

print(f"Dataset Version: {meta.get('version')}")
print(f"Input Dimension: {meta.get('input_dim')}")
print(f"Action Classes: {list(meta['action_classes'].keys())}")
print(f"Procedure Steps: {meta['num_steps']}")
print(f"Train/Val/Test Sizes: {meta['splits']}")

train_data = np.load(os.path.join(DATA_DIR, 'train', 'samples.npz'))
print(f"Train features shape: {train_data['features'].shape}")
assert train_data['features'].shape[-1] == 512, "Feature dimension must be 512"

# Distributions
def print_dist(labels, name):
    valid_labels = labels[labels != -100]
    unique, counts = np.unique(valid_labels, return_counts=True)
    print(f"\n{name} Distribution:")
    for u, c in zip(unique, counts):
        print(f"  Class {u}: {c}")

print_dist(train_data['action'], "Current Action")
print_dist(train_data['step'], "Procedure Step")
print_dist(train_data['next_action'], "Next Action")
print_dist(train_data['anomaly'], "Anomaly")



## 5. Dataset Sanity Check

In [ ]:
print("--- SANITY CHECK EXAMPLES ---")
actions_inv = {v: k for k, v in meta['action_classes'].items()}

for i in range(3):
    length = train_data['lengths'][i]
    acts = train_data['action'][i][:length]
    anom = train_data['anomaly'][i][:length]
    next_a = train_data['next_action'][i][:length]
    
    is_anom = (anom > 0).any()
    anom_str = "ANOMALOUS" if is_anom else "NORMAL"
    
    trans = []
    last_a = None
    for a in acts:
        act_name = actions_inv[a]
        if act_name != last_a:
            trans.append(act_name)
            last_a = act_name
            
    print(f"\nExample {i} ({anom_str}): {' -> '.join(trans)}")
    if is_anom:
        anom_idx = np.where(anom > 0)[0][0]
        print(f"  Anomaly triggered at frame {anom_idx} during action: {actions_inv[acts[anom_idx]]}")
        print(f"  Expected next action: {actions_inv[next_a[anom_idx]]}")



## 6. Model Definition

In [ ]:
# Import REAL model from repository
import sys
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

try:
    from ml.models.temporal.model import CausalGRUFallback
except ImportError:
    print("ERROR: Could not import ml.models.temporal.model. Make sure the codebase is in /content/ml")

model_wrapper = CausalGRUFallback(
    input_dim=512,
    hidden_dim=256,
    num_layers=2,
    num_actions=meta['num_actions'],
    num_steps=meta['num_steps'],
    dropout=0.1
)
model_wrapper.device = device
model_wrapper.build()
model = model_wrapper.model

print("Model Architecture:")
print(model)

# Verify output dict
dummy_input = torch.randn(2, 60, 512).to(device)
out = model(dummy_input)
print(f"\nInput: [2, 60, 512]")
print(f"Outputs:")
for k, v in out.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {v.shape}")



## 7. DataLoader & Loss

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

class TemporalDataset(Dataset):
    def __init__(self, npz_path):
        d = np.load(npz_path)
        self.features = d['features']
        self.action = d['action']
        self.step = d['step']
        self.next_action = d['next_action']
        self.anomaly = d['anomaly']
        self.lengths = d['lengths']
        
    def __len__(self):
        return len(self.features)
        
    def __getitem__(self, idx):
        # We feed random sub-windows during training
        L = self.lengths[idx]
        if L < 16:
            end_t = L - 1
        else:
            end_t = np.random.randint(15, L)
            
        seq = self.features[idx][:end_t+1]
        
        act = self.action[idx][end_t]
        step = self.step[idx][end_t]
        n_act = self.next_action[idx][end_t]
        anom = self.anomaly[idx][end_t]
        
        padded_seq = np.zeros((60, 512), dtype=np.float32)
        valid_len = min(len(seq), 60)
        padded_seq[-valid_len:] = seq[-valid_len:]
        
        return {
            'features': torch.tensor(padded_seq, dtype=torch.float32),
            'action': torch.tensor(act, dtype=torch.long),
            'step': torch.tensor(step, dtype=torch.long),
            'next_action': torch.tensor(n_act, dtype=torch.long),
            'anomaly': torch.tensor(anom, dtype=torch.float32)
        }

train_dataset = TemporalDataset(os.path.join(DATA_DIR, 'train', 'samples.npz'))
val_dataset = TemporalDataset(os.path.join(DATA_DIR, 'val', 'samples.npz'))
test_dataset = TemporalDataset(os.path.join(DATA_DIR, 'test', 'samples.npz'))

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Loss functions (No artificial balancing applied per instructions)
criterion_action = nn.CrossEntropyLoss()
criterion_step = nn.CrossEntropyLoss()
criterion_anomaly = nn.BCELoss()

# Weights
W_ACTION = 1.0
W_STEP = 1.0
W_NEXT = 1.0
W_ANOMALY = 2.0

def compute_loss(out, batch):
    loss_a = criterion_action(out['action_logits'], batch['action'].to(device))
    loss_s = criterion_step(out['step_logits'], batch['step'].to(device))
    loss_n = criterion_action(out['next_action_logits'], batch['next_action'].to(device))
    loss_an = criterion_anomaly(out['anomaly_score'], batch['anomaly'].to(device))
    
    total = W_ACTION*loss_a + W_STEP*loss_s + W_NEXT*loss_n + W_ANOMALY*loss_an
    return total, loss_a, loss_s, loss_n, loss_an



## 8. Training Loop

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
import copy

optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=5)

num_epochs = 80
best_val_loss = float('inf')
best_model_wts = copy.deepcopy(model.state_dict())

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        out = model(batch['features'].to(device))
        loss, la, ls, ln, lan = compute_loss(out, batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            out = model(batch['features'].to(device))
            loss, _, _, _, _ = compute_loss(out, batch)
            val_loss += loss.item()
            
    train_loss /= len(train_loader)
    val_loss /= len(val_loader)
    scheduler.step(val_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        print("  -> New best model!")

model.load_state_dict(best_model_wts)



## 9. Validation & Test Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

def evaluate(loader, name="Validation"):
    model.eval()
    all_preds_a, all_trues_a = [], []
    all_preds_s, all_trues_s = [], []
    all_preds_n, all_trues_n = [], []
    all_preds_an, all_trues_an = [], []
    
    with torch.no_grad():
        for batch in loader:
            out = model(batch['features'].to(device))
            
            all_preds_a.extend(out['action_logits'].argmax(dim=1).cpu().numpy())
            all_trues_a.extend(batch['action'].numpy())
            
            all_preds_s.extend(out['step_logits'].argmax(dim=1).cpu().numpy())
            all_trues_s.extend(batch['step'].numpy())
            
            all_preds_n.extend(out['next_action_logits'].argmax(dim=1).cpu().numpy())
            all_trues_n.extend(batch['next_action'].numpy())
            
            all_preds_an.extend((out['anomaly_score'] > 0.5).int().cpu().numpy())
            all_trues_an.extend(batch['anomaly'].int().numpy())
            
    print(f"\n{'='*40}\n{name} Evaluation\n{'='*40}")
    print("--- Current Action ---")
    print(classification_report(all_trues_a, all_preds_a))
    
    print("--- Procedure Step ---")
    print(classification_report(all_trues_s, all_preds_s))
    
    print("--- Next Action ---")
    print(classification_report(all_trues_n, all_preds_n))
    
    print("--- Anomaly ---")
    print(classification_report(all_trues_an, all_preds_an))
    
evaluate(val_loader, "Validation")
evaluate(test_loader, "Test")



## 10. Temporal Rolling Test (Procedure Evaluation)

In [ ]:
print("\n--- TEMPORAL ROLLING TEST (Whole Episode) ---")

d = np.load(os.path.join(DATA_DIR, 'test', 'samples.npz'))

for test_idx in [0, 1, 2]:
    seq = d['features'][test_idx]
    length = d['lengths'][test_idx]
    anomalies = d['anomaly'][test_idx][:length]
    true_actions = d['action'][test_idx][:length]
    
    is_anom = (anomalies > 0).any()
    
    true_trans = []
    last_t = None
    for a in true_actions:
        a_n = actions_inv[a]
        if a_n != last_t:
            true_trans.append(a_n)
            last_t = a_n
            
    print(f"\n{'='*60}")
    print(f"Rolling Evaluation for Sequence {test_idx} (Anomalous: {is_anom})")
    print(f"Expected sequence: {' -> '.join(true_trans)}")
    print(f"{'='*60}")
    
    predicted_seq = []
    
    for t in range(0, length, 5):
        window = seq[max(0, t-59):t+1]
        
        padded = np.zeros((1, 60, 512), dtype=np.float32)
        padded[0, -len(window):] = window
        
        with torch.no_grad():
            out = model(torch.tensor(padded).to(device))
            
        pred_act = actions_inv[out['action_logits'].argmax(dim=1).item()]
        pred_step = out['step_logits'].argmax(dim=1).item()
        pred_next = actions_inv[out['next_action_logits'].argmax(dim=1).item()]
        pred_anom = out['anomaly_score'].item()
        
        print(f"  Frame {t:3d} | Act: {pred_act:7s} | Step: {pred_step:2d} | Next: {pred_next:7s} | Anom: {pred_anom:.2f}")
        
        if not predicted_seq or predicted_seq[-1] != pred_act:
            predicted_seq.append(pred_act)
            
    print(f"\nPredicted transition sequence: {' -> '.join(predicted_seq)}")



## 11. Checkpoint Packaging

In [ ]:
import zipfile

out_dir = 'TemporalGRU-ASTRA-v1'
os.makedirs(out_dir, exist_ok=True)

torch.save({
    'model_state_dict': model.state_dict(),
    'classes': list(meta['action_classes'].keys()),
    'input_dim': 512,
    'num_actions': meta['num_actions'],
    'num_steps': meta['num_steps']
}, os.path.join(out_dir, 'best.pt'))

with open(os.path.join(out_dir, 'README.md'), 'w') as f:
    f.write("# TemporalGRU-ASTRA-v1\nPlace this in `checkpoints/temporal/TemporalGRU-ASTRA-v1/best.pt`\n")

shutil.make_archive(out_dir, 'zip', out_dir)
print(f"Saved {out_dir}.zip. Download this file to your local machine.")

from google.colab import files
try:
    files.download(f"{out_dir}.zip")
except:
    pass

